# Validate dataset

Purpose:

- Verify processed WAVs match metadata and meet expected sample rate/duration/channel constraints;
- Produce a shor CSV report of issues for manual correction.

(It's just a validation notebook if data is included outside the notebook 03 or other different things that might happen)

In [ ]:
import csv
from pathlib import Path

import soundfile as sf

PROCESSED_DIR = Path("data/processed/wavs")
METADATA_CSV = Path("data/processed/metadata.csv")
TARGET_SAMPLE_RATE = 22050
MIN_DURATION = 0.2
MAX_DURATION = 20.0
REPORT_PATH = Path("data/outputs/validation_report.csv")

## What do we check here?

- Each metadata row maps to a processed WAV file (filename exits);
- WAV file readable and has expected sample rate (TARGET_SAMPLE_RATE);
- Channels == 1 (mono);
- Duration is within `MIN_DURATION` and `MAX_DURATION`;
- Metadata transcription is non-empty (so model has text to learn);
- Any read errors are reported so you can inspect corrupted files.


In [ ]:
metadata_rows = []

if not METADATA_CSV.exists():
    print(f"Metadata file not found: {METADATA_CSV}. Run preprocessing first.")
else:
    with open(METADATA_CSV, "r", encoding="utf-8") as file:
        for line_raw in file:
            line = line_raw.rstrip("\n")
            if not line:
                continue
            parts = [part.strip() for part in line.split("|")]
            if len(parts) == 1:
                parts = [parts[0], "", ""]
            elif len(parts) == 2:
                parts.append("")
            metadata_rows.append((parts[0], parts[1], parts[2]))

print(f"Loaded {len(metadata_rows)} metadata rows from {METADATA_CSV}")

## Validation

Validation loop, core checks and collect issues.

In [ ]:
issues = []
checked = 0

for filename, transcription, speaker in metadata_rows:
    checked += 1
    file_path = PROCESSED_DIR / filename

    if not file_path.exits():
        issues.append(
            (
                filename,
                "MISSING_FILEM",
                "",
                "File missing in processed dir",
                transcription,
            )
        )
        continue

    try:
        info = sf.info(str(file_path))
        sample_rate = int(info.samplerate) if info.samplerate else None
        frames = int(info.frames) if info.frames else 0
        duration = round(frames / sample_rate, 3) if sample_rate else None
        channel = int(info.channels) if info.channels else None
    except Exception as error:
        issues.append((filename, "READ_ERROR", str(error), "", transcription))
        continue

    if sample_rate is None:
        issues.append(
            (
                filename,
                "NO_SAMPLE_RATE",
                "",
                "Could not read sample rate",
                transcription,
            )
        )
    elif sample_rate != TARGET_SAMPLE_RATE:
        issues.append(
            (
                filename,
                "SAMPLE_RATE_MISMATCH",
                sample_rate,
                TARGET_SAMPLE_RATE,
                transcription,
            )
        )

    if duration is None or duration < MIN_DURATION:
        issues.append(
            (filename, "TOO_SHORT", duration, f"<{MIN_DURATION}", transcription)
        )
    if duration and duration > MAX_DURATION:
        issues.append(
            (filename, "TOO_LONG", duration, f">{MAX_DURATION}", transcription)
        )

    if channel is not None and channel != 1:
        issues.append((filename, "CHANNELS", channel, 1, transcription))

    if not transcription:
        issues.append(
            (
                filename,
                "MISSING_TRANSCRIPT",
                "",
                "Fill transcription in metadata.csv",
                transcription,
            )
        )

print(
    f"Validation checks complete. Metadata rows checked: {checked}. Issues found: {len(issues)}"
)

## Write Report

Write a CSV report and show a short preview

In [ ]:
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(REPORT_PATH, "w", encoding="utf-8", newline="") as out_file:
    writer = csv.writer(out_file)
    writer.writerow(["filename", "issue", "value", "expectation", "transcription"])
    for issue in issues:
        writer.writerow(issue)

print(f"Report written: {REPORT_PATH} (issues: {len(issues)})")

if issues:
    print("First 20 issues (preview):")
    for issue in issues[:20]:
        print(" ", issue)
else:
    print("No issues found - dataset looks consistent. Proceed to training smoke test.")

## How to act on the report

- Open `data/outputs/validation_report.csv` and fix listed problems;
    - `MISSING_FILE`: re-run preprocessing or copy missing processed WAVs into `data/processed/wavs/`;
    - `SAMPLE_RATE_MISMATCH`: reprocess the source with the target sample rate in `03_preprocessing.ipynb`;
    - `TOO_SHORT` / `TOO_LONG`: inspect and re-record or split long utterances;
    - `CHANNELS`: covert to mono in preprocessing;
    - `MISSING_TRANSCRIPT`: fill the transcription column in `data/processed/metadata.csv`;
- After fixes, re-run this notebook.